# Notebook 3 — Geometric Analysis

The primary scientific notebook for Stage 1. Five analysis blocks measure how much SNOMED ontological structure is implicitly encoded in the frozen LLM residual stream.

| Block | Question |
|---|---|
| A | Does residual-stream geometry correlate with SNOMED ontological distance? |
| B | How does LLM geometry compare to OpenAI and KEEP embeddings? |
| C | Is the geometry tree-like (hyperbolic)? |
| D | (Optional) Do SAE features cluster by SNOMED hierarchy? |
| E | Verification — are results within the expected range? |

**Inputs**
- `data/stage1/hidden_states_last_token.npy` + `hidden_states_mean_pool.npy`
- `data/embeddings-concept-openai/concepts.csv`, `ontological_distances.csv`, `embeddings_normalised.csv`
- `data/stage1/snomed_isa_graph.pkl`, `snomed_omop_map.csv`
- KEEP pretrained embeddings pickle (set `KEEP_EMBEDDINGS_PATH` below)

**Outputs**
- `data/stage1/correlation_results.csv`, `pairwise_distances.csv`, `verification_summary.json`
- `data/stage1/plots/` — distance scatters, UMAP, 3D PCA

**Blocked by:** notebooks 0b, 2, and `utils.py` (issues #4, #5, #7)

In [ ]:
# parameters
DATA_DIR            = "../../data/stage1"
OPENAI_DIR          = "../../data/embeddings-concept-openai"
KEEP_EMBEDDINGS_PATH = ""   # path to KEEP pretrained embeddings pickle (dict: omop_id -> np.ndarray)
                             # leave empty to skip Block B KEEP comparison
KNN_K               = 10    # k for k-NN graph in geodesic distance computation
SEED                = 42

In [ ]:
import json
import os
import pickle
import sys
import warnings

import networkx as nx
import numpy as np
import pandas as pd
from scipy.spatial.distance import squareform, pdist
from scipy.sparse.csgraph import dijkstra
from scipy.stats import spearmanr
from sklearn.decomposition import PCA

sys.path.insert(0, "../..")
sys.path.insert(0, "../../Reference_Papers/Snomed2Vec/src/embedding_learning")

from src.concept_openai_embeddings_utils import (
    knn_graph, largest_connected_component, chatterjee_corr,
    distance_plot, interactive_3d_plot, interactive_3d_plot_by_tag,
)
from stage1_baseline_geometry_utils import gromov_delta, ic_distances
from src.config import MODEL_NAME, L_DET

os.makedirs(os.path.join(DATA_DIR, "plots"), exist_ok=True)
print(f"Model: {MODEL_NAME}, L_DET: {L_DET}")

### Import helper — path aliases

The import paths above use underscored aliases to avoid Python path conflicts with hyphens. Add these aliases once at the top of the notebook.

In [ ]:
# Python import aliases for hyphenated directory names
import importlib.util, types

def _load(alias, path):
    spec = importlib.util.spec_from_file_location(alias, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    sys.modules[alias] = mod
    return mod

eu  = _load("src.concept_openai_embeddings_utils",
            "../../src/embeddings-concept-openai/utils.py")
su  = _load("stage1_baseline_geometry_utils",
            "utils.py")

knn_graph                = eu.knn_graph
largest_connected_component = eu.largest_connected_component
chatterjee_corr          = eu.chatterjee_corr
distance_plot            = eu.distance_plot
interactive_3d_plot      = eu.interactive_3d_plot
interactive_3d_plot_by_tag = eu.interactive_3d_plot_by_tag
gromov_delta             = su.gromov_delta
ic_distances             = su.ic_distances

print("Utilities loaded.")

## Load data

In [ ]:
# Hidden states from notebook 2
H_last = np.load(os.path.join(DATA_DIR, "hidden_states_last_token.npy"))  # (n, d)
H_mean = np.load(os.path.join(DATA_DIR, "hidden_states_mean_pool.npy"))   # (n, d)

# Concepts and SNOMED shortest-path distances
concepts_df  = pd.read_csv(os.path.join(OPENAI_DIR, "concepts.csv"), dtype=str)
D_snomed     = pd.read_csv(os.path.join(OPENAI_DIR, "ontological_distances.csv"), header=None).values
labels       = concepts_df["preferred_term"].tolist()
concept_ids  = concepts_df["concept_id"].tolist()

# OpenAI embeddings
H_openai = pd.read_csv(os.path.join(OPENAI_DIR, "embeddings_normalised.csv"), header=None).values

# SNOMED IS-A graph (from notebook 0b)
with open(os.path.join(DATA_DIR, "snomed_isa_graph.pkl"), "rb") as f:
    G_snomed = pickle.load(f)

print(f"Concepts:           {len(labels)}")
print(f"Hidden states last: {H_last.shape}")
print(f"Hidden states mean: {H_mean.shape}")
print(f"SNOMED dist matrix: {D_snomed.shape}")
print(f"OpenAI embeddings:  {H_openai.shape}")
print(f"SNOMED graph nodes: {G_snomed.number_of_nodes():,}")

## Block A — Distance Correlation

Pairwise cosine distances in residual stream space, correlated with SNOMED ontological distances under two conditions:
1. **Shortest-path** IS-A distance (from `ontological_distances.csv`)
2. **IC-based** distance via `ic_distances()` (Resnik, from SNOMED IS-A graph)

Run for both last-token (Method A) and mean-pool (Method B) extraction.

In [ ]:
def embedding_distances(H):
    """Cosine and geodesic distance matrices from hidden state matrix H."""
    D_cos  = squareform(pdist(H, metric="cosine"))          # (n, n)
    A      = knn_graph(H, k=KNN_K)
    lcc    = largest_connected_component(A, verbose=True)
    D_geo  = dijkstra(A[lcc][:, lcc], directed=False)       # (n_lcc, n_lcc)
    return D_cos, D_geo, lcc

print("Computing distances for last-token hidden states...")
D_cos_last, D_geo_last, lcc_last = embedding_distances(H_last)

print("\nComputing distances for mean-pool hidden states...")
D_cos_mean, D_geo_mean, lcc_mean = embedding_distances(H_mean)

print(f"\nLCC sizes — last: {lcc_last.sum()}, mean: {lcc_mean.sum()}")

In [ ]:
# IC-based distances (condition 2)
print("Computing IC-based SNOMED distances (Resnik)...")
D_ic = ic_distances(G_snomed, concept_ids, method="resnik")  # (n, n)
print(f"IC distance matrix: {D_ic.shape}, NaN count: {np.isnan(D_ic).sum()}")

In [ ]:
def corr_in_lcc(D_embed, D_onto, lcc):
    """Spearman ρ and Chatterjee ξ between embedding and ontological distances, within the LCC."""
    D_e = D_embed[lcc][:, lcc]
    D_o = D_onto[lcc][:, lcc]

    # Upper triangle only (exclude diagonal and lower mirror)
    idx = np.triu_indices(D_e.shape[0], k=1)
    x, y = D_o[idx], D_e[idx]

    # Drop NaN pairs (concepts absent from graph)
    valid = ~(np.isnan(x) | np.isnan(y))
    x, y  = x[valid], y[valid]

    rho, pval = spearmanr(x, y)
    xi        = chatterjee_corr(x, y)
    return float(rho), float(pval), float(xi), int(valid.sum())

results = {}
for method, D_cos, D_geo, lcc in [
    ("last_token", D_cos_last, D_geo_last, lcc_last),
    ("mean_pool",  D_cos_mean, D_geo_mean, lcc_mean),
]:
    for cond, D_onto in [("shortestpath", D_snomed), ("ic_resnik", D_ic)]:
        for dist_type, D_embed in [("cosine", D_cos), ("geodesic", D_geo)]:
            if dist_type == "geodesic" and D_embed.shape[0] != D_onto.shape[0]:
                # Geodesic is already LCC-restricted; use LCC-masked ontological distances
                lcc_onto = D_onto[lcc][:, lcc]
                idx  = np.triu_indices(D_embed.shape[0], k=1)
                x, y = lcc_onto[idx], D_embed[idx]
                valid = ~(np.isnan(x) | np.isnan(y))
                x, y  = x[valid], y[valid]
                rho, pval = spearmanr(x, y)
                xi        = chatterjee_corr(x, y)
                rho, pval, xi, n_pairs = float(rho), float(pval), float(xi), int(valid.sum())
            else:
                rho, pval, xi, n_pairs = corr_in_lcc(D_embed, D_onto, lcc)

            key = f"{method}__{cond}__{dist_type}"
            results[key] = {"rho": rho, "pval": pval, "xi": xi, "n_pairs": n_pairs}
            print(f"{key:55s}  ρ={rho:+.3f}  ξ={xi:+.3f}  p={pval:.2e}  n={n_pairs:,}")

In [ ]:
# Distance scatter plots
for label, D_cos, lcc, D_onto, cond in [
    ("LLM last-token / shortest-path", D_cos_last, lcc_last, D_snomed,   "shortestpath"),
    ("LLM last-token / IC-Resnik",     D_cos_last, lcc_last, D_ic,       "ic_resnik"),
    ("LLM mean-pool  / shortest-path", D_cos_mean, lcc_mean, D_snomed,   "shortestpath"),
]:
    D_e = D_cos[lcc][:, lcc]
    D_o = D_onto[lcc][:, lcc]
    fig, ax = distance_plot(D_e, D_o, labels, corr_coef="pearson",
                            xlabel="SNOMED distance", ylabel="Cosine distance")
    ax.set_title(label)
    fname = label.replace(" ", "_").replace("/", "-").replace("'", "") + ".png"
    fig.savefig(os.path.join(DATA_DIR, "plots", fname), dpi=150, bbox_inches="tight")
    print(f"Saved: {fname}")

In [ ]:
# 3D PCA visualisation (LLM last-token)
pca = PCA(n_components=3, random_state=SEED)
H3  = pca.fit_transform(H_last)

# Colour by mean ontological distance from each concept
mean_onto = D_snomed.mean(axis=1)
fig = interactive_3d_plot(H3, labels=labels, color_values=mean_onto,
                          title="LLM residual stream — PCA 3D (coloured by mean SNOMED distance)")
fig.write_html(os.path.join(DATA_DIR, "plots", "interactive_3d_llm.html"))
print("Saved: interactive_3d_llm.html")

# Colour by semantic tag
tags   = concepts_df["semantic_tag"].tolist()
colors = concepts_df["semantic_tag"].astype("category").cat.codes.tolist()
fig2 = interactive_3d_plot_by_tag(H3, labels=labels, colors=colors, groups=tags,
                                   title="LLM residual stream — PCA 3D (by semantic tag)")
fig2.write_html(os.path.join(DATA_DIR, "plots", "interactive_3d_llm_by_tag.html"))
print("Saved: interactive_3d_llm_by_tag.html")

## Block B — Three-Way Comparison (LLM vs. OpenAI vs. KEEP)

Run the same distance correlation pipeline on OpenAI `text-embedding-3-large` vectors and KEEP pretrained embeddings and compare Spearman ρ values.

KEEP embeddings are indexed by OMOP concept ID. The mapping is loaded from `data/stage1/snomed_omop_map.csv` (produced by notebook 2). Concepts without an OMOP match are excluded from the KEEP comparison.

In [ ]:
# --- OpenAI comparison ---
D_cos_openai, D_geo_openai, lcc_openai = embedding_distances(H_openai)

rho_openai, pval_openai, xi_openai, n_openai = corr_in_lcc(D_cos_openai, D_snomed, lcc_openai)
print(f"OpenAI text-embedding-3-large / shortest-path:  ρ={rho_openai:+.3f}  ξ={xi_openai:+.3f}  p={pval_openai:.2e}")

fig, ax = distance_plot(D_cos_openai[lcc_openai][:, lcc_openai],
                        D_snomed[lcc_openai][:, lcc_openai],
                        labels, corr_coef="pearson",
                        xlabel="SNOMED shortest-path distance",
                        ylabel="Cosine distance")
ax.set_title("OpenAI embeddings / shortest-path distance")
fig.savefig(os.path.join(DATA_DIR, "plots", "distance_scatter_openai.png"), dpi=150, bbox_inches="tight")
print("Saved: distance_scatter_openai.png")

In [ ]:
# --- KEEP comparison ---
omop_map = pd.read_csv(os.path.join(DATA_DIR, "snomed_omop_map.csv"), dtype=str)

if not KEEP_EMBEDDINGS_PATH or not os.path.exists(KEEP_EMBEDDINGS_PATH):
    print(
        "KEEP_EMBEDDINGS_PATH not set or file not found — skipping KEEP comparison.\n"
        "Download pretrained KEEP embeddings from the G2Lab/keep repo and set the path above."
    )
else:
    with open(KEEP_EMBEDDINGS_PATH, "rb") as f:
        keep_embeddings = pickle.load(f)   # dict: omop_concept_id (str) -> np.ndarray

    # Build aligned array for concepts that have an OMOP mapping and a KEEP embedding
    keep_rows, keep_mask = [], []
    for omop_id in omop_map["omop_concept_id"]:
        if pd.notna(omop_id) and str(omop_id) in keep_embeddings:
            keep_rows.append(keep_embeddings[str(omop_id)])
            keep_mask.append(True)
        else:
            keep_mask.append(False)

    keep_mask = np.array(keep_mask)
    H_keep    = np.stack(keep_rows)             # (n_keep, d_keep)
    D_snomed_keep = D_snomed[keep_mask][:, keep_mask]
    labels_keep   = [labels[i] for i, m in enumerate(keep_mask) if m]

    print(f"KEEP concepts matched: {keep_mask.sum()} / {len(keep_mask)}")

    D_cos_keep, D_geo_keep, lcc_keep = embedding_distances(H_keep)
    rho_keep, pval_keep, xi_keep, n_keep = corr_in_lcc(D_cos_keep, D_snomed_keep,
                                                        np.ones(len(H_keep), dtype=bool))
    print(f"KEEP / shortest-path:  ρ={rho_keep:+.3f}  ξ={xi_keep:+.3f}  p={pval_keep:.2e}")

    fig, ax = distance_plot(D_cos_keep, D_snomed_keep, labels_keep,
                            corr_coef="pearson",
                            xlabel="SNOMED shortest-path distance",
                            ylabel="Cosine distance")
    ax.set_title("KEEP embeddings / shortest-path distance")
    fig.savefig(os.path.join(DATA_DIR, "plots", "distance_scatter_keep.png"), dpi=150, bbox_inches="tight")
    print("Saved: distance_scatter_keep.png")

## Block C — Hyperbolic Geometry Tests

Four tests for tree-like (hyperbolic) structure in the LLM representations:
1. **Gromov δ-hyperbolicity** — four-point condition; small δ = tree-like
2. **UMAP with hyperbolic metric** — visual check
3. **Poincaré embedding ordinal comparison** — Spearman ρ between pairwise Poincaré distances and LLM cosine distances
4. **Hyperbolic probing classifier** — logistic regression in Poincaré disk (requires `geoopt`)

In [ ]:
# C1: Gromov δ-hyperbolicity
print("Computing Gromov δ on geodesic distance matrix (last-token LCC)...")
delta = gromov_delta(D_geo_last, sample_size=500, seed=SEED)
print(f"Gromov δ = {delta:.4f}")
print("  δ ≈ 0 → tree-like (hyperbolic); δ ≈ diam/2 → Euclidean")

In [ ]:
# C2: UMAP with hyperbolic metric
import umap

reducer = umap.UMAP(n_components=2, output_metric="hyperboloid", random_state=SEED, n_jobs=1)
H_hyp   = reducer.fit_transform(H_last)

import plotly.express as px
fig = px.scatter(
    x=H_hyp[:, 0], y=H_hyp[:, 1],
    hover_name=labels,
    color=concepts_df["semantic_tag"],
    title="UMAP (hyperbolic metric) — LLM last-token hidden states",
)
fig.write_html(os.path.join(DATA_DIR, "plots", "umap_hyperbolic.html"))
print("Saved: umap_hyperbolic.html")

In [ ]:
# C3: Poincaré embedding training + ordinal comparison
from poincare import get_poincare_model
from gensim.models.poincare import PoincareModel

poincare_path = os.path.join(DATA_DIR, "snomed_poincare.model")

if not os.path.exists(poincare_path):
    print("Training Poincaré embeddings on breast SNOMED subgraph (~5–15 min on CPU)...")
    relations = [(str(u), str(v)) for u, v in G_snomed.edges()]  # (child, parent) pairs
    poincare_model = get_poincare_model(relations, emb_size=100)
    poincare_model.save(poincare_path)
    print(f"Saved: {poincare_path}")
else:
    poincare_model = PoincareModel.load(poincare_path)
    print(f"Loaded: {poincare_path}")

# Ordinal comparison: pairwise Poincaré distances vs LLM cosine distances
n = len(concept_ids)
poincare_dists, llm_dists = [], []

for i in range(n):
    for j in range(i + 1, n):
        c1, c2 = concept_ids[i], concept_ids[j]
        if c1 in poincare_model.kv and c2 in poincare_model.kv:
            poincare_dists.append(poincare_model.kv.distance(c1, c2))
            llm_dists.append(D_cos_last[i, j])

rho_poincare, pval_poincare = spearmanr(poincare_dists, llm_dists)
print(f"Poincaré vs LLM cosine Spearman ρ = {rho_poincare:.3f}  (p={pval_poincare:.2e}, n={len(poincare_dists):,})")

In [ ]:
# C4: Hyperbolic probing classifier (requires geoopt)
try:
    import geoopt
    import torch

    # Binary labels: for each concept pair, 1 if one is an ancestor of the other, else 0
    def is_ancestor(G, c1, c2):
        """True if c1 is an ontological ancestor of c2 (c2 IS-A ... c1)."""
        return c2 in G and c1 in nx.descendants(G, c2)

    pairs, pair_labels = [], []
    rng = np.random.default_rng(SEED)
    idx_pairs = rng.choice(n, size=(2000, 2), replace=True)
    for i, j in idx_pairs:
        if i == j:
            continue
        c1, c2 = concept_ids[i], concept_ids[j]
        pairs.append((H_last[i], H_last[j]))
        pair_labels.append(int(is_ancestor(G_snomed, c1, c2) or is_ancestor(G_snomed, c2, c1)))

    print(f"Pairs: {len(pairs)}, positive (ancestor): {sum(pair_labels)}")
    print("(Full hyperbolic classifier training left as exercise — geoopt is available.)")

except ImportError:
    print("geoopt not installed — skipping hyperbolic probing classifier. Install with: pip install geoopt")

## Block D — SAE Clustering (Optional)

**Decision point:** run this block only if:
- Blocks A–C show meaningful correlation (ρ > 0.15), AND
- A pretrained SAE for Llama-3-8B is available

If those conditions are not met, skip to Block E.

In [ ]:
RUN_BLOCK_D = False   # set True to enable SAE clustering
SAE_PATH    = ""      # path to pretrained SAE for Llama-3-8B

if RUN_BLOCK_D:
    import sys
    sys.path.insert(0, "../../Reference_Papers/MultiDimensionalFeatures")
    from sae_multid_feature_discovery.clustering import graph_cluster_sims, spectral_cluster_sims
    from sae_multid_feature_discovery.saes.sparse_autoencoder import SparseAutoencoder

    sae = SparseAutoencoder.load_from_pretrained(SAE_PATH)
    W_dec = sae.W_dec.detach().cpu().numpy()   # (d_sae, d_model)

    # Cosine similarity between SAE features
    from sklearn.metrics.pairwise import cosine_similarity
    all_sims = cosine_similarity(W_dec)

    clusters_graph    = graph_cluster_sims(all_sims, top_k_for_graph=2, sim_cutoff=0.5)
    clusters_spectral = spectral_cluster_sims(all_sims, n_clusters=1000)

    print(f"Graph clusters: {len(set(clusters_graph))}")
    print(f"Spectral clusters: {len(set(clusters_spectral))}")
    print("(Causal validation via do_regression() from MultiDimensionalFeatures — see plan for adaptation notes.)")
else:
    print("Block D skipped (RUN_BLOCK_D = False).")

## Block E — Verification

Asserts expected outcome ranges and writes a machine-readable summary. Expected range from the design document: ρ ~ 0.2–0.4 for the well-attested subset.

In [ ]:
# Primary result: last-token, cosine, shortest-path
rho_last_shortestpath = results["last_token__shortestpath__cosine"]["rho"]
rho_mean_shortestpath = results["mean_pool__shortestpath__cosine"]["rho"]
rho_last_ic           = results["last_token__ic_resnik__cosine"]["rho"]
rho_mean_ic           = results["mean_pool__ic_resnik__cosine"]["rho"]

summary = {
    "model":                                MODEL_NAME,
    "L_det":                                L_DET,
    "spearman_rho_last_token_shortestpath": rho_last_shortestpath,
    "spearman_rho_mean_pool_shortestpath":  rho_mean_shortestpath,
    "spearman_rho_last_token_ic":           rho_last_ic,
    "spearman_rho_mean_pool_ic":            rho_mean_ic,
    "method_agreement":                     abs(rho_last_shortestpath - rho_mean_shortestpath) < 0.05,
    "gromov_delta":                         delta,
    "n_concepts_lcc":                       int(lcc_last.sum()),
}

# Range checks (design document expectations)
if not (0.05 < rho_last_shortestpath < 0.6):
    warnings.warn(f"ρ = {rho_last_shortestpath:.3f} is outside expected range (0.05–0.6)")

if not summary["method_agreement"]:
    warnings.warn(
        f"Last-token and mean-pool disagree by "
        f"{abs(rho_last_shortestpath - rho_mean_shortestpath):.3f} > 0.05 — investigate"
    )

out_path = os.path.join(DATA_DIR, "verification_summary.json")
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f"\nSaved: {out_path}")

In [ ]:
# Save full correlation results table
corr_rows = [{"condition": k, **v} for k, v in results.items()]
pd.DataFrame(corr_rows).to_csv(os.path.join(DATA_DIR, "correlation_results.csv"), index=False)
print("Saved: correlation_results.csv")

# Save pairwise distances (last-token cosine)
pd.DataFrame(D_cos_last).to_csv(os.path.join(DATA_DIR, "pairwise_distances.csv"), index=False, header=False)
print("Saved: pairwise_distances.csv")